# Faza testowa - załącznik

Czyta gotowe katalogi przebiegów z `results/runs/` oraz pliki adnotacji i bufor segmentów, i składa z nich rozbicia szczegółowe: wszystkie miary wszystkich konfiguracji, podziały po podzbiorach zapytań, tabelę przejść i wykaz materiału testowego. Nie liczy nic na nagraniach i nie ładuje żadnego modelu. Bloki bez przebiegu wypisują, czego brakuje. Tabele główne są w `test_results.ipynb` i `test_summary.ipynb`.

**Wymaga:** tego samego co `test_results.ipynb`, czyli przebiegów wszystkich konfiguracji testowych. Wykaz materiału potrzebuje dodatkowo bufora segmentów zamrożonej strategii:

```powershell
python scripts/run_segmentation.py configs/e1b_office.yaml configs/e1b_tbbt.yaml
```

Recall@K i mAP w procentach, przecinek dziesiętny. Miary drugorzędne podawane są bez przedziałów i bez orzekania o istotności: wnioskowanie prowadzone jest wyłącznie dla miary głównej (`recall@10`), a porównywanie kilku miar naraz zwiększałoby ryzyko, że różnica okaże się pozorna.

**Zapisuje:** nic, tylko wypisuje.

In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import datasets
from src.data import queries as queries_module
from src.evaluation import compare, tables
from src.evaluation import runs as runs_module
from src.segmentation import segments as seg
from src.segmentation.build import WHOLE_CLIP
from src.utils import experiments as exp
from src.utils import frozen as frozen_module
from src.utils.vocabulary import COMPLEXITY_VALUES, REQUIREMENT_TAGS

for module in (datasets, queries_module, compare, runs_module, seg, tables, exp,
               frozen_module):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory
load_queries = queries_module.load_queries

SPLIT = "test"
METRIC = "recall@10"

DATASETS = list(exp.DATASETS)
LABEL = exp.DATASET_NAMES
BASE, FULL = exp.BASE, exp.FULL
SIGNALS = list(exp.ADDED_SIGNALS)
SIGNAL_NAME = exp.SIGNAL_NAMES

FROZEN = frozen_module.load_frozen()
# the strategy every test collection is built with, from the verdict itself
FROZEN_STRATEGY = FROZEN.segmentation.strategy if FROZEN.segmentation else None

# The order of chapter 6: the baseline, the single-signal variants, the full
# pipeline and its two families. Three blocks below walk it, so it stands here
# rather than in the first of them.
ORDER = ([exp.BASE] + list(exp.ADDS_SIGNAL) + ["E4-D", exp.FULL]
         + [exp.no_label(signal) for signal in exp.ADDED_SIGNALS]
         + [exp.swap_label(signal) for signal in exp.SWAPPED_SIGNALS])

runs = runs_module.load_runs(SPLIT)


def available(labels):
    """Runs of the given labels, on the datasets ALL of them cover."""
    present, shared, skipped = runs_module.available(runs, labels)
    for label, reason in skipped:
        print(f"  {label}: {reason}")
    return runs_module.select(runs, present, shared), shared


def missing(labels):
    """Prints what has to be run for a block to have anything to show."""
    gaps = [label for label in labels if label not in runs]
    if gaps:
        print(f"brak przebiegow: {', '.join(gaps)} - patrz naglowek notatnika")
    return bool(gaps)


def winner_label(signal):
    """The variant of a signal the full pipeline carries, or None."""
    candidates = [label for label, added in exp.ADDS_SIGNAL.items() if added == signal]
    if len(candidates) == 1:
        return candidates[0]
    decision = getattr(FROZEN, signal, None)
    if decision is None:
        return None
    return next((label for label in candidates
                 if exp.LABEL_MECHANISM.get(label) == decision.chosen), None)


print(f"przebiegi znalezione dla czesci {SPLIT!r}: {len(runs)} etykiet")
for label in sorted(runs):
    print(f"  {label:<24}{', '.join(sorted(runs[label]))}")
if not runs:
    print("  (jeszcze zadnego)")
print(f"\nzamrozona strategia segmentacji: {FROZEN_STRATEGY or '? - uzupelnij frozen.yaml'}")


## 1. Wszystkie miary wszystkich konfiguracji

Jeden wiersz na konfigurację i zbiór, kolumny to wszystkie miary, które przebieg policzył: R@1, R@5, R@10, mAP, mediana i średnia rangi pierwszego trafienia. Liczby pochodzą z `metrics.json` przebiegu, więc są dokładnie tymi, które zostały zapisane, a nie przeliczone tutaj.

Kolejność wierszy jest kolejnością rozdziału 6: najpierw BAZA, potem warianty pojedynczych sygnałów, potok pełny i jego rodziny.

In [ ]:
MEASURES = ["recall@1", "recall@5", "recall@10", "mAP", "MedR", "MnR"]
SHARES = {"recall@1", "recall@5", "recall@10", "mAP"}

rows, seen = [], []
for label in ORDER:
    if label in seen or label not in runs:
        continue
    seen.append(label)
    for dataset in DATASETS:
        run = runs[label].get(dataset)
        if run is None:
            continue
        overall = run["metrics"].get("overall", {})
        rows.append([exp.display(label, base_as_name=False), LABEL[dataset],
                     overall.get("query_count", "-")]
                    + [tables.percent(overall.get(measure)) if measure in SHARES
                       else tables.number(overall.get(measure), 1)
                       for measure in MEASURES])

absent = [label for label in ORDER if label not in seen]
if rows:
    tables.show("Wszystkie miary wszystkich konfiguracji",
                ["Konfiguracja", "Zbior", "Zapytan"] + MEASURES, rows,
                align="ll" + "r" * (len(MEASURES) + 1),
                note="Wartosci wprost z metrics.json przebiegu. MedR i MnR to mediana "
                     "i srednia rangi pierwszego trafnego fragmentu, liczone po "
                     "zapytaniach, ktore trafienie mialy. Bez przedzialow: "
                     "wnioskowanie prowadzone jest wylacznie dla recall@10.")
else:
    print("brak przebiegow")
if absent:
    print(f"\nbez przebiegu: {', '.join(absent)}")


## 2. Recall@10 w podziale na podzbiory zapytań

Dla każdej konfiguracji i każdego podzbioru: R@10 na zapytaniach należących do podzbioru ("tak") i na pozostałych ("nie"), z wierszem liczebności. Podzbiory pochodzą z `compare.subset_ids`, te same, których używają kontrasty, więc liczba tutaj i liczba w tabelach głównych opisują ten sam zbiór zapytań.

Podzbiór pusty znaczy "ten zbiór nie przypisuje tego znacznika" i wypisuje się jako `-`, nigdy jako zero.

In [ ]:
SUBSETS = ([(f"requirements:{tag}", tag) for tag in REQUIREMENT_TAGS]
           + [(f"complexity:{value}", f"zlozonosc {value}")
              for value in COMPLEXITY_VALUES]
           + [("identities:>=1", "co najmniej jedna postac"),
              ("identities:>=2", "dwie lub wiecej postaci")])

for dataset in DATASETS:
    present = [label for label in ORDER if dataset in runs.get(label, {})]
    if not present:
        print(f"{LABEL[dataset]}: brak przebiegow")
        continue

    header, sizes = ["Konfiguracja"], []
    usable = []
    for spec, described in SUBSETS:
        try:
            inside, outside = compare.subset_ids(dataset, SPLIT, spec)
        except (FileNotFoundError, ValueError):
            continue
        if not inside:
            continue
        usable.append((described, inside, outside))
        header += [f"{described}: tak", f"{described}: nie"]
        sizes += [len(inside), len(outside)]
    if not usable:
        print(f"{LABEL[dataset]}: zaden podzbior nie jest tu przypisany")
        continue

    rows = [["liczba zapytan"] + sizes]
    for label in present:
        per_query = runs[label][dataset]["per_query"]
        cells = [exp.display(label, base_as_name=False)]
        for _, inside, outside in usable:
            cells += [tables.percent(compare.query_mean(per_query, METRIC, inside)),
                      tables.percent(compare.query_mean(per_query, METRIC, outside))]
        rows.append(cells)

    tables.show(f"{LABEL[dataset]}: R@10 w podziale na podzbiory",
                header, rows,
                align="l" + "r" * (len(header) - 1),
                note="Wiersz 'liczba zapytan' jest mianownikiem obu kolumn kazdego "
                     "podzbioru. Podzbior nieprzypisany w tym zbiorze nie ma tu "
                     "kolumn wcale - nie ma ich jako zer.")


## 3. VATEX według słownika Kinetics-400

Klipy VATEX-a dzielą się na te, których klasa Kinetics-600 należy do 400 nazw, na których uczono SlowFasta, i na pozostałe. Poza tymi 400 nazwami dopasowana fraza czynnościowa nie ma na czym wylądować, więc obie strony pytają sygnał ruchu o co innego. Wiersz aktywacji mówi, jak często sygnał w ogóle się w każdej z nich odzywał.

Podział dotyczy wyłącznie VATEX-a: tylko klip niesie etykietę Kinetics.

In [ ]:
DATASET_K400 = "vatex"
present = [label for label in ORDER if DATASET_K400 in runs.get(label, {})]

if not present:
    print(f"brak przebiegow na zbiorze {LABEL[DATASET_K400]}")
else:
    try:
        inside, outside = compare.subset_ids(DATASET_K400, SPLIT, "kinetics_vocab:yes")
    except FileNotFoundError as problem:
        inside = outside = None
        print(f"{problem}\nbez vatex_split_test.csv nie ma czego dzielic")

    if inside is not None:
        # `list | str` is the signature of tables.render: a row given as a
        # plain string becomes a subheading, which is what the second half
        # of this table is. Without it a linter reads the append below as a
        # mistake and "fixing" it would drop the heading.
        rows: list[list | str] = [["liczba zapytan", len(inside),
                                   len(outside)]]
        for label in present:
            per_query = runs[label][DATASET_K400]["per_query"]
            rows.append([exp.display(label, base_as_name=False),
                         tables.percent(compare.query_mean(per_query, METRIC, inside)),
                         tables.percent(compare.query_mean(per_query, METRIC, outside))])

        rows.append("aktywacja sygnalu ruchu [%]")
        for label in present:
            run = runs[label][DATASET_K400]
            share_in = compare.activation_share(run, inside).get("motion")
            share_out = compare.activation_share(run, outside).get("motion")
            if share_in is None and share_out is None:
                continue          # this configuration carries no motion signal
            rows.append([exp.display(label, base_as_name=False),
                         tables.percent(share_in), tables.percent(share_out)])

        # The unconditional split above hides the one difference there is: the two
        # groups differ above all in how often the motion signal speaks (the
        # activation rows), not in what it does when it speaks. So its cost is
        # also compared on the queries it spoke about, group by group.
        sf_run = runs.get("E2-C", {}).get(DATASET_K400)
        base_run = runs.get(BASE, {}).get(DATASET_K400)
        if sf_run is not None and base_run is not None:
            spoke = compare.active_ids(sf_run, "motion")
            rows.append("E2-C wobec BAZY na zapytaniach z wlaczonym ruchem")
            rows.append(["liczba zapytan", len(inside & spoke), len(outside & spoke)])
            for label, run in ((BASE, base_run), ("E2-C", sf_run)):
                rows.append([f"R@10 {exp.display(label, base_as_name=False)}"] + [
                    tables.percent(compare.query_mean(run["per_query"], METRIC,
                                                      ids & spoke))
                    for ids in (inside, outside)])
            rows.append(["delta R@10 [p.p.]"] + [
                tables.points(compare.query_mean(sf_run["per_query"], METRIC, ids & spoke)
                              - compare.query_mean(base_run["per_query"], METRIC,
                                                   ids & spoke))
                for ids in (inside, outside)])
            # How the loss on these queries splits between the two groups: a
            # group's mean change times its number of queries, over the same sum
            # for both. The larger mean loss sits in the smaller group, so the
            # two shares below are what says whether it matters in total.
            lost = [len(ids & spoke)
                    * (compare.query_mean(sf_run["per_query"], METRIC, ids & spoke)
                       - compare.query_mean(base_run["per_query"], METRIC, ids & spoke))
                    for ids in (inside, outside)]
            rows.append(["udzial w zapytaniach z ruchem [%]"] + [
                tables.number(100 * len(ids & spoke) / len(spoke), 1)
                for ids in (inside, outside)])
            rows.append(["udzial w lacznej stracie [%]"] + [
                tables.number(100 * part / sum(lost), 1) for part in lost])

        tables.show("VATEX wedlug slownika Kinetics-400",
                    ["Konfiguracja", "klasa w K400", "klasa spoza K400"], rows,
                    note="Gorna czesc to R@10, dolna - udzial zapytan, dla ktorych "
                         "sygnal ruchu mial co powiedziec; ostatni blok - roznica E2-C "
                         "wobec BAZY tylko na tych zapytaniach. Konfiguracja "
                         "bez sygnalu ruchu nie ma wiersza w czesci aktywacji.")


## 4. Aktywacja według znaczników

Ten sam udział co w zestawieniu aktywacji, ale w pełnym rozbiciu: dla potoku pełnego, sygnał po sygnale, znacznik po znaczniku. Kolumna "wszystkie" jest punktem odniesienia - sygnał aktywny równie często wewnątrz i na zewnątrz swojego znacznika nie jest bramkowany tym, czym miał być.

Sygnał bramkowany jest wyłączany z ważenia zapytania, dla którego nie ma nic do powiedzenia, więc zasięg jest połową tego, co znaczy wkład.

In [ ]:
if not missing([FULL]):
    for dataset in DATASETS:
        run = runs.get(FULL, {}).get(dataset)
        if run is None:
            continue
        everywhere = compare.activation_share(run)
        usable = []
        for tag in REQUIREMENT_TAGS:
            try:
                inside, _ = compare.subset_ids(dataset, SPLIT, f"requirements:{tag}")
            except (FileNotFoundError, ValueError):
                continue
            if inside:
                usable.append((tag, inside))
        if not usable:
            print(f"{LABEL[dataset]}: brak przypisanych znacznikow")
            continue

        rows = []
        for signal in SIGNALS:
            if dataset not in exp.datasets_of(signal):
                continue
            targets = set(exp.target_tags(signal))
            cells = [SIGNAL_NAME[signal], tables.percent(everywhere.get(signal))]
            for tag, inside in usable:
                share = compare.activation_share(run, inside).get(signal)
                mark = "*" if tag in targets else " "
                cells.append(f"{tables.percent(share)}{mark}")
            rows.append(cells)

        tables.show(f"{LABEL[dataset]}: aktywacja wedlug znacznikow",
                    ["Sygnal", "wszystkie"] + [tag for tag, _ in usable], rows,
                    align="l" + "r" * (len(usable) + 1),
                    note="Gwiazdka oznacza znacznik docelowy tego sygnalu - ten, "
                         "ktorego eksperyment sygnal rozstrzyga (vocabulary."
                         "TAG_EXPERIMENT). Sygnal nieobecny na zbiorze nie ma wiersza.")


## 5. Tabela przejść

Wkład indywidualny każdego wariantu w rozbiciu na zapytania naprawione ($0\to1$), zepsute ($1\to0$) i niezmienione. Wiersz nazywa komponent, nie sygnał: odkąd stoją w tabeli oba warianty, musi mówić, czy chodzi o YOLO11, czy o YOLOE-11.

Kolumna "blisko progu" liczy zapytania, które zmieniły przynależność do pierwszej dziesiątki, mając poprawny fragment w obu przebiegach na randze 7-14 (`compare.BORDERLINE_BAND`), czyli takie, które zmieniły status przez samo położenie progu odcięcia, a nie przez to, że wariant coś znalazł.

Różnica dwóch pierwszych kolumn podzielona przez `n` daje wkład liczony po zapytaniach; tabele główne liczą go dla seriali po odcinkach, stąd rozbieżność do 0,4 p.p.

In [ ]:
# every variant that adds one signal to the base, the frozen one and the
# rejected one alike -- the development phase decided between them, but the test
# runs exist for both and the appendix reports what each one did.
INDIVIDUAL = [label for label in ORDER if label in exp.ADDS_SIGNAL] + ["E4-D"]


def named(label):
    """Row label: the experiment, and the component this variant puts on the base.

    Built rather than taken from LABEL_NAMES so that E4-D reads like every other
    row of the table -- it too is the base plus one component, and a row that
    left out "BAZA +" would look like a different kind of comparison.
    """
    return f"{label}: {exp.BASE_DISPLAY} + {exp.LABEL_COMPONENT[label]}"


rows = []
for label in INDIVIDUAL:
    candidate, reference = (label, BASE)
    group, columns = available([candidate, reference])
    if len(group) < 2:
        # eight columns, and the row has to carry all eight or render() cannot
        # measure the table
        rows.append([named(label), *["-"] * 8])
        continue
    for dataset in columns:
        after, before = group[candidate][dataset], group[reference][dataset]
        counts = compare.transitions(after, before)
        total = sum(counts.values())
        rows.append([named(label), LABEL[dataset], total,
                     counts["0->1"], counts["1->0"], counts["1->1"], counts["0->0"],
                     compare.borderline(after, before),
                     tables.points((counts["0->1"] - counts["1->0"]) / total
                                   if total else None)])

if rows:
    tables.show("Przejscia przy wkladzie indywidualnym",
                ["Konfiguracja", "Zbior", "n", "0->1", "1->0", "1->1", "0->0",
                 "Blisko progu", "Delta_ind [p.p.]"], rows, align="ll" + "r" * 7,
                note="Cztery grupy sa rozlaczne i wyczerpujace, wiec ich suma jest "
                     "kolumna n. Zapytanie, na ktore odpowiedzial tylko jeden z dwoch "
                     "przebiegow, jest odmawiane, nie pomijane - inaczej n bylaby "
                     "mniejsza, a wygladalaby na cala kolekcje. Blisko progu: ranga "
                     f"poprawnego fragmentu w przedziale {compare.BORDERLINE_BAND[0]}-"
                     f"{compare.BORDERLINE_BAND[1]} w obu przebiegach naraz.")
else:
    print("brak par przebiegow do porownania")

## 5a. Przejścia przy pozostałych układach wkładu

To samo rozbicie dla dwóch układów, których porównaniem jest potok pełny: wkład krańcowy (pełny wobec pełnego bez sygnału) i efekt podmiany (pełny wobec pełnego z drugim mechanizmem tego sygnału). Wartość dodatnia przemawia w obu za konfiguracją zamrożoną.

Wiersz nazywa mechanizm, który potok pełny niesie; nazwa pochodzi z `configs/frozen.yaml`, więc tabela nie może się rozjechać z zamrożonym potokiem.

In [ ]:
def carried(signal):
    """The component the FULL pipeline carries for one signal, from the verdict."""
    label = winner_label(signal)
    return exp.LABEL_COMPONENT.get(label, SIGNAL_NAME[signal]) if label else "?"


def swapped_in(signal):
    """The component the swap pipeline puts in its place."""
    decision = getattr(FROZEN, signal, None)
    if decision is None:
        return "?"
    return next((exp.LABEL_COMPONENT[label]
                 for label, mechanism in exp.LABEL_MECHANISM.items()
                 if exp.ADDS_SIGNAL.get(label) == signal
                 and mechanism == decision.rejected), "?")


#: the adjective the appendix uses for a signal removed from the pipeline:
#: "PELNY bez sygnalu obiektowego". Kept beside SIGNAL_NAMES rather than derived
#: from it, because Polish declension is not a string operation.
WITHOUT_SIGNAL = {"caption": "opisowego", "objects": "obiektowego", "motion": "ruchu",
                  "face_regions": "regionow", "identity": "tozsamosci"}

# (candidate, reference, layout, description); the candidate is the FULL pipeline
# in both layouts, so a positive value speaks for the frozen configuration. The
# wording is the one the appendix already uses, so a number in the thesis can be
# found here by the name of its row.
COMPARISONS = (
    [(*exp.marginal_pair(signal), "krancowy",
      f"PELNY bez sygnalu {WITHOUT_SIGNAL[signal]}")
     for signal in SIGNALS]
    + [(*exp.swap_pair(signal), "podmiana",
        f"PELNY z {swapped_in(signal)} zamiast {carried(signal)}")
       for signal in exp.SWAPPED_SIGNALS])

rows = []
for candidate, reference, layout, described in COMPARISONS:
    group, columns = available([candidate, reference])
    if len(group) < 2:
        continue
    for dataset in columns:
        after, before = group[candidate][dataset], group[reference][dataset]
        counts = compare.transitions(after, before)
        total = sum(counts.values())
        rows.append([layout, described, LABEL[dataset], total,
                     counts["0->1"], counts["1->0"], counts["1->1"], counts["0->0"],
                     compare.borderline(after, before),
                     tables.points((counts["0->1"] - counts["1->0"]) / total
                                   if total else None)])

if rows:
    tables.show("Przejscia przy pozostalych ukladach wkladu",
                ["Uklad", "Porownanie", "Zbior", "n", "0->1", "1->0", "1->1", "0->0",
                 "Blisko progu", "Delta [p.p.]"], rows, align="lll" + "r" * 7,
                note="Porownywany jest potok PELNY, wiec wartosc dodatnia przemawia "
                     "za konfiguracja zamrozona. W ukladzie krancowym wiersz nazywa "
                     "sygnal usuwany z potoku, w ukladzie podmiany - "
                     "mechanizm zamrozony i ten, ktory go zastepuje. "
                     "Kolumny jak w tabeli wyzej.")
else:
    print("brak par przebiegow do porownania")

## 6. Wykaz materiału - wiersze testowe

Czas korpusu, liczba fragmentów zamrożonej strategii i liczba zapytań, po jednym wierszu na zbiór, dla części testowej. Wiersze części deweloperskiej daje `e1_segmentation.ipynb`; ten blok liczy tę samą tabelę dla drugiej części, żeby obie powstawały z tego samego bufora segmentów.

Czas korpusu jest mierzony na osi treści: sekundy masek przejścia i czerni nie liczą się, bo nie liczą się też dla żadnego sygnału. Bierze się on z kolumny `duration` bufora segmentów, która niesie długość na osi treści; `file_duration` (czyli `end - start`) niesie długość na osi pliku i te dwie różnią się dokładnie tam, gdzie maska wpada w środek fragmentu. Liczba nagrań pochodzi z tego samego bufora, więc wszystkie kolumny opisują ten sam materiał.

In [ ]:
if FROZEN_STRATEGY is None:
    print("configs/frozen.yaml nie niesie jeszcze werdyktu E1 - brak strategii")
else:
    # A VATEX clip IS the fragment, so its collection is built under whole_clip
    # whatever E1 decided; only the series go through the frozen strategy.
    #
    # Built HERE and not in the configuration cell: there the verdict may still
    # be missing and every series would map to None, so the annotation below
    # would be a wish. In this branch the verdict exists -- that is what the
    # `if` above established -- and the type follows from the code.
    strategy_of: dict[str, str] = {}
    for dataset in DATASETS:
        strategy_of[dataset] = FROZEN_STRATEGY if dataset in exp.SERIES else WHOLE_CLIP

    rows = []
    for dataset in DATASETS:
        try:
            collection = load_queries(dataset, SPLIT)
        except FileNotFoundError as problem:
            print(f"{LABEL[dataset]}: {problem}")
            continue

        path = datasets.segments_csv(strategy_of[dataset], dataset)
        fragments = [row for row in seg.load(path)
                     if row["split"] == SPLIT] if path.exists() else []
        # `duration` is the content length and `file_duration` is end - start;
        # they differ exactly where a mask or a black stretch falls inside a
        # fragment, and the table of the thesis is the content one
        lengths = [row["duration"] for row in fragments]

        rows.append([LABEL[dataset], strategy_of[dataset],
                     len({row["episode"] for row in fragments}) or "-",
                     tables.number(sum(lengths) / 3600, 2) if lengths else "-",
                     len(fragments) or "-",
                     tables.number(sum(lengths) / len(lengths), 1) if lengths else "-",
                     len(collection)])

    tables.show(f"Material czesci testowej, strategia {FROZEN_STRATEGY} "
                f"({exp.STRATEGY_NAMES.get(FROZEN_STRATEGY, FROZEN_STRATEGY)})",
                ["Zbior", "Strategia", "Nagran", "Czas korpusu [h]", "Fragmentow",
                 "Sr. dlugosc [s]", "Zapytan"], rows, align="ll" + "r" * 5,
                note="Czas mierzony na osi tresci: sekundy maski przejscia i czerni "
                     "sie nie licza. Klip VATEX-a jest fragmentem sam w sobie, wiec "
                     "jego kolekcja powstaje pod whole_clip niezaleznie od werdyktu "
                     "E1. Myslnik w kolumnie fragmentow znaczy, ze bufor segmentow tej "
                     "strategii nie obejmuje jeszcze czesci testowej - uruchom "
                     "scripts/run_segmentation.py.")
